<a href="https://colab.research.google.com/github/BrownParticleAstro/dmtools_notebooks/blob/main/colab_plot_logic_20261010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A very quick refactor using Github Copilot to break the plot logic away from the web app logic.

In [ ]:
import io
import base64
from typing import Dict, List, Optional, Tuple, Any
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.ticker import LogFormatterMathtext
import numpy as np

In [ ]:
# ============================================================================
# CORE: Context-agnostic plot rendering (works anywhere)
# ============================================================================

async def render_plot_from_data(
    plot_data: Dict[str, Any],
    plot_properties: Dict[str, Any],
    display_data_list: List[Dict[str, Any]],
    figsize: Tuple[int, int] = (10, 10),
    output_format: str = "base64"  # "base64", "file", "object"
) -> Tuple[str, str]:
    """
    Context-agnostic plot renderer.

    Args:
        plot_data: The nested plot structure (passed in, not fetched)
        plot_properties: Plot config (xUnits, xMin, xMax, yMin, yMax, plotType, etc.)
        display_data_list: List of display dicts with style/color and data arrays
        figsize: Figure size tuple
        output_format: How to return the plot ("base64" for web, "file" for disk, "object" for Colab)

    Returns:
        (plot_name, plot_output) where plot_output depends on output_format
    """
    px = 1 / 100

    # Create figure
    fig = plt.figure(figsize=figsize, linewidth=0, edgecolor='#D0D6DB', facecolor='#D0D6DB')
    gs = gridspec.GridSpec(64, 64)
    ax = fig.add_subplot(gs[2:63, 2:63])
    ax.set_facecolor('white')
    ax.tick_params(axis='both', labelsize=16)

    plot_name = plot_properties.get('name', 'Plot')
    plot_id = plot_data.get("plot_node", {}).get("record", {}).get("id", -1)

    # Iterate over displays and their data
    for display in display_data_list:
        display_properties = display.get('properties', {})
        style = display_properties.get('style', 'line')
        color = display_properties.get('color', 'black')

        # Get matplotlib kwargs from your style library
        line_plot_kwargs = get_style_mpl(color, style)
        line_plot_kwargs.pop('style', None)
        line_plot_kwargs.pop('fill', None)

        for data in display.get("data", []):
            data_properties = data.get('properties', {})

            y_rescale = float(data_properties.get('yRescale', 1))
            x_rescale = float(data_properties.get('xRescale', 1))
            x_units = data_properties.get('xUnits', 'GeV/c^2')

            trace_name = data_properties.get('label_short', 'Trace')
            list_data = data_properties.get('values', [[[0.0, 0.0], [0.0, 1.0]]])

            # Plot each trace
            for trace in list_data:
                x = [float(item[0]) * x_rescale for item in trace]
                y = [float(item[1]) * y_rescale for item in trace]

                selected_unit = plot_properties.get('xUnits', 'GeV/c^2')
                plot_type = plot_properties.get('plotType')

                if plot_type == "Cross Section vs WIMP Mass":
                    unit_x = [convert_mass_units(val, x_units, selected_unit) for val in x]
                elif plot_type == "Cross Section / Mass [in GeV] vs Mass[GeV]":
                    unit_x = [convert_mass_units(val, x_units, 'GeV') for val in x]
                    y = [(float(item[1]) * y_rescale) / xi for item, xi in zip(trace, unit_x)]
                else:
                    unit_x = [convert_mass_units(val, x_units, selected_unit) for val in x]

                ax.plot(unit_x, y, **line_plot_kwargs)

                if style == 'fill':
                    fill_plot_kwargs = get_style_mpl(color, style)
                    fill_plot_kwargs.pop('style', None)
                    fill_plot_kwargs.pop('marker', None)
                    fill_plot_kwargs.pop('markersize', None)
                    fill_plot_kwargs.pop('linestyle', None)
                    fill_plot_kwargs.pop('linewidth', None)
                    fill_plot_kwargs.pop('fill', None)
                    ax.fill_between(unit_x, y, **fill_plot_kwargs)

    # Axes configuration
    ax.set_xscale('log')
    ax.set_yscale('log')

    ymin_exp = float(plot_properties.get('yMin', -42)) if plot_properties.get('yMin') else -42
    ymax_exp = float(plot_properties.get('yMax', -42)) if plot_properties.get('yMax') else -42
    xmin = float(plot_properties.get('xMin', 0)) if plot_properties.get('xMin') else 0
    xmax = float(plot_properties.get('xMax', 3)) if plot_properties.get('xMax') else 10000

    ax.set_xlim([xmin, xmax])
    ax.set_ylim([10**ymin_exp, 10**ymax_exp])

    ax.set_ylabel(r"$\mathrm{Cross\ Section}\ [cm^{2}]\ (\mathrm{normalized\ to\ nucleon})$", fontsize=18)
    selected_unit = plot_properties.get('xUnits', 'GeV/c^2')
    ax.set_xlabel(get_x_label(selected_unit), fontsize=18)

    plot_title_default = r"$\mathrm{WIMP\ Mass\ vs\ Cross\ Section\ (Plot\ ID:\ " + str(plot_id) + r")}$"
    plot_title = plot_name + ' (' + str(plot_id) + ')' if plot_name else plot_title_default
    ax.set_title(plot_title, fontsize=18)

    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.yaxis.set_major_formatter(LogFormatterMathtext())
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname('DejaVu Sans')

    # Output handling
    if output_format == "base64":
        img_svg = io.BytesIO()
        fig.savefig(img_svg, format='svg', dpi=100, bbox_inches='tight', pad_inches=0.2)
        img_svg.seek(0)
        plot_output = base64.b64encode(img_svg.getvalue()).decode('utf8')

    elif output_format == "file":
        plot_output = f"/tmp/plot_{plot_id}.svg"
        fig.savefig(plot_output, format='svg', dpi=100, bbox_inches='tight', pad_inches=0.2)

    elif output_format == "object":
        # Return the matplotlib figure object (great for Colab display())
        plot_output = fig

    else:
        raise ValueError(f"Unknown output_format: {output_format}")

    if output_format != "object":
        plt.close(fig)

    return plot_name, plot_output

In [ ]:
# ============================================================================
# WEB ADAPTER: Wrapper for your existing Quart app
# ============================================================================

async def set_plot_data(conn, plot_id, data_id=-1, schema="data", output_format="base64"):
    """
    Web-friendly wrapper that bridges database + cache layer to core renderer.
    """
    plot_id_int = int(plot_id)

    # Fetch from your DB/cache
    plot_data, status_code = await get_plot_display_data_nodes_cached(
        conn, plot_id_int, data_id=data_id, schema=schema
    )

    if status_code != 200:
        plot_name, dmtools_edit_plot_url, dmtools_edit_legend_url = await create_empty_chart_and_legend_svgs()
        return plot_name, dmtools_edit_plot_url, dmtools_edit_legend_url

    plot_data_cache = await cache_get("dmtools_current_plot")
    plot_properties = plot_data_cache.get("plot_node", {}).get("properties", {})
    display_data_list = plot_data_cache.get("display_data", [])

    # Render using context-agnostic function
    plot_name, plot_output = await render_plot_from_data(
        plot_data=plot_data_cache,
        plot_properties=plot_properties,
        display_data_list=display_data_list,
        output_format=output_format
    )

    # Cache for web (if needed)
    if output_format == "base64":
        await cache_set("dmtools_plot_url", plot_output)

    return plot_name, plot_output




In [ ]:
# ============================================================================
# COLAB ADAPTER: Standalone usage in notebooks
# ============================================================================

async def plot_from_dict(plot_dict: Dict[str, Any]) -> None:
    """
    Quick plot rendering for Colab/Jupyter without database.

    Usage in Colab:
    ```python
    plot_dict = {
        "plot_node": {"record": {"id": 1}, "properties": {"name": "My Plot", ...}},
        "display_data": [{
            "properties": {"style": "line", "color": "blue"},
            "data": [{"properties": {"values": [[[1, 2], [2, 4]]], ...}, ...}]
        }, ...]
    }
    await plot_from_dict(plot_dict)
    ```
    """
    plot_properties = plot_dict.get("plot_node", {}).get("properties", {})
    display_data_list = plot_dict.get("display_data", [])

    plot_name, fig = await render_plot_from_data(
        plot_data=plot_dict,
        plot_properties=plot_properties,
        display_data_list=display_data_list,
        output_format="object"
    )

    plt.show()


# ============================================================================
# STANDALONE UTILITY: Plot from raw arrays (Colab-first)
# ============================================================================

def quick_plot(
    x_data: List[float],
    y_data: List[float],
    title: str = "Plot",
    xlabel: str = "X",
    ylabel: str = "Y",
    style: str = "line",
    color: str = "blue",
    figsize: Tuple[int, int] = (10, 8),
    log_scale: bool = True
) -> None:
    """
    Minimal Colab-friendly plotting function (no DB, no cache).

    Usage:
    ```python
    quick_plot([1, 10, 100], [1e-40, 1e-42, 1e-44], log_scale=True)
    ```
    """
    fig, ax = plt.subplots(figsize=figsize)

    line_plot_kwargs = get_style_mpl(color, style)
    line_plot_kwargs.pop('style', None)
    line_plot_kwargs.pop('fill', None)

    ax.plot(x_data, y_data, **line_plot_kwargs)

    if log_scale:
        ax.set_xscale('log')
        ax.set_yscale('log')

    ax.set_title(title, fontsize=16)
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    ax.grid(True, alpha=0.3)

    plt.show()